# 🎓 캡스톤 — 미니 신경망을 처음부터 (ch1~12 총정리)

1부의 졸업 작품입니다. **새 개념 없이**, ch1~12에서 배운 조각만 조립해
**진짜 신경망**을 NumPy 로 밑바닥부터 만들고 학습시킵니다.
(웹 챕터가 없는 **노트북 전용** 종합 실습이에요.)

**이 노트북에서 하는 것**
1. **문제** — 안쪽 뭉치 vs 바깥 고리: 직선으로는 절대 못 나누는 데이터
2. **순전파** — 신경망 = 행렬곱의 연쇄 (ch3) + 비선형 활성화 (ch5)
3. **손실** — 오차 제곱의 평균: 최소제곱(ch10)·손실(ch12)의 재출연
4. **역전파** — 전치행렬(ch3) 곱으로 책임을 거꾸로 배분 + 수치미분 검산 (ch12)
5. **학습** — 경사하강 $W \leftarrow W - \eta \nabla L$ (ch12): 정확도 47% → **100%**
6. **확인** — 은닉층이 공간을 어떻게 구부렸는지 내적(ch2)·투영(ch10)·PCA(ch8)로 들여다보기

> 밖에서 받아 적는 건 딱 **두 줄**(시그모이드·tanh 의 기울기 공식)뿐 — 유도는 2부 미적분에서.
> 나머지는 전부 이미 배운 것들입니다. 정말인지 확인하러 가요.

> 🤖 **AI 튜터** — 아래 셀을 실행하면 웹앱처럼 화면 오른쪽 아래에 🤖 버튼이 생깁니다.
> 버튼을 누르면 채팅창이 열리고, 어디를 스크롤해도 그 자리에 떠 있어요. 막히면 언제든 질문하세요.
> (로컬 Ollama 필요 — 꺼져 있으면 켜는 방법이 안내되고, 튜터 없이도 학습에는 지장 없습니다.)

In [ ]:
# 🤖 AI 튜터 — 실행하면 오른쪽 아래에 🤖 버튼이 떠요 (웹앱과 같은 로컬 AI, 클릭해서 열기)
from tutor import chat

chat('capstone')

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

# 한글 폰트 설정 (그래프의 한글이 깨지지 않게)
plt.rcParams['font.family'] = 'Malgun Gothic'
plt.rcParams['axes.unicode_minus'] = False

print('numpy', np.__version__, '— 준비 완료')

## 1. 문제 — 직선으로 못 나누는 두 무리

**분류(classification)** = 새 점이 들어오면 어느 무리인지 맞히는 문제.
스팸 판별, 사진 속 고양이 찾기, 악성 종양 진단이 전부 이것입니다.

과녁 모양 데이터를 만들어요 — **안쪽 뭉치**(정답 0)와 **바깥 고리**(정답 1).

- 점 하나 = 벡터 $(x_1, x_2)$ — ch1 그대로
- 점 200개를 쌓으면 **데이터 행렬 $X$ (200×2)** — ch3의 "행 = 샘플, 열 = 특징"
- 정답을 모은 $y$ (200×1) — **레이블(label)** 이라고 불러요. 여기선 0 아니면 1.

이 데이터의 핵심: **어떤 직선을 그어도 두 무리를 나눌 수 없습니다.**
4장의 연립방정식도, 10장의 최소제곱도 결국 "직선"이 한계였죠. 오늘 그 벽을 넘습니다.

In [ ]:
rng = np.random.default_rng(42)
n = 100                                   # 무리당 점 개수

# 안쪽 뭉치 — 반지름 0~1.1 안 아무 방향 (cos·sin 은 ch5 회전에서 본 그 코사인·사인)
r = rng.uniform(0, 1.1, n); theta = rng.uniform(0, 2 * np.pi, n)
inner = np.c_[r * np.cos(theta), r * np.sin(theta)]

# 바깥 고리 — 반지름 1.9~2.8 의 도넛 모양
r = rng.uniform(1.9, 2.8, n); theta = rng.uniform(0, 2 * np.pi, n)
outer = np.c_[r * np.cos(theta), r * np.sin(theta)]

X = np.vstack([inner, outer])                       # (200, 2) 데이터 행렬 — 행 = 점, 열 = 좌표
y = np.vstack([np.zeros((n, 1)), np.ones((n, 1))])  # (200, 1) 레이블 — 안쪽 0, 바깥 1

plt.figure(figsize=(5.2, 5.2))
plt.scatter(*inner.T, s=20, color='#4f46e5', label='안쪽 무리 (y=0)')
plt.scatter(*outer.T, s=20, color='#f43f5e', label='바깥 고리 (y=1)')
plt.gca().set_aspect('equal')
plt.title('이 두 무리, 직선 하나로 나눌 수 있을까? (불가능!)')
plt.legend(); plt.show()

print('X:', X.shape, ' y:', y.shape, '— 점 = 벡터, 전체 = 행렬. 늘 하던 그대로!')

## 2. 순전파 — 신경망은 행렬곱의 연쇄

오늘 만들 신경망(2층)의 전체 흐름:

$$\underbrace{X}_{200\times 2} \;\xrightarrow{W_1,\ b_1}\; \underbrace{Z_1}_{200\times 8} \;\xrightarrow{\tanh}\; \underbrace{A_1}_{200\times 8} \;\xrightarrow{W_2,\ b_2}\; \underbrace{Z_2}_{200\times 1} \;\xrightarrow{\sigma}\; \underbrace{\hat{y}}_{200\times 1}$$

수식으로는 네 줄이 전부입니다:

$$Z_1 = XW_1 + b_1, \qquad A_1 = \tanh(Z_1), \qquad Z_2 = A_1 W_2 + b_2, \qquad \hat{y} = \sigma(Z_2)$$

**처음 보는 것들 정리** (여기서 다 설명하고 넘어가요)

| 기호 | 읽는 법 | 뜻 |
|---|---|---|
| 은닉층 | hidden layer | 입력과 출력 사이의 중간 층. 여기선 뉴런 8개 |
| $W_1$(2×8), $W_2$(8×1) | 가중치 행렬 | ch3의 신경망 층 $Wx+b$ 그대로 — 학습으로 바뀌는 값 |
| $\tanh$ | 탄치 (하이퍼볼릭 탄젠트) | 아무 숫자나 −1~1 로 눌러주는 S자 곡선 — **활성화 함수** |
| $\sigma$ | 시그모이드 | 아무 숫자나 0~1 로 눌러주는 S자 곡선: $\sigma(z) = \frac{1}{1+e^{-z}}$. 출력을 "바깥 고리일 확률"로 읽게 해줘요 |
| $\hat{y}$ | 와이 햇 | 모델의 **예측값**. 모자(^) = "추정치" 표시 — 정답 $y$ 와 구분 |

**왜 중간에 tanh 를 끼울까?** — ch5에서 심어둔 복선의 회수입니다. 활성화 없이 행렬곱만 쌓으면

$$(XW_1)W_2 = X(W_1W_2)$$

행렬곱의 결합법칙(ch3) 때문에 **행렬 하나로 접혀버립니다.** 백 층을 쌓아도 선형변환 1개 = 직선 경계뿐.
층 사이에 비선형(tanh)을 끼워야 공간을 **구부릴** 수 있어요.

뉴런 하나가 하는 일도 이미 아는 것: 입력과 자기 가중치의 **내적**(ch2) + 편향. 그게 전부입니다.

In [ ]:
rng = np.random.default_rng(7)
W1 = rng.normal(0, 0.8, (2, 8)); b1 = np.zeros((1, 8))   # 1층: 입력 2차원 → 뉴런 8개
W2 = rng.normal(0, 0.8, (8, 1)); b2 = np.zeros((1, 1))   # 2층: 뉴런 8개 → 출력 1개

def sigmoid(z):
    return 1 / (1 + np.exp(-z))     # σ — 아무 숫자나 0~1 로

def forward(X, W1, b1, W2, b2):
    """순전파 — 입력이 확률이 될 때까지. 행렬곱 두 번이 전부."""
    Z1 = X @ W1 + b1        # 1층: 행렬곱 + 편향 (ch3 의 Wx+b)
    A1 = np.tanh(Z1)        # 비선형 활성화 (ch5) — 없으면 행렬 하나로 접힌다
    Z2 = A1 @ W2 + b2       # 2층
    y_hat = sigmoid(Z2)     # 0~1 확률로
    return Z1, A1, Z2, y_hat

# 학습 전 상태를 복사해 보관 — 마지막에 학습 전/후를 비교하려고
W1_0, b1_0, W2_0, b2_0 = W1.copy(), b1.copy(), W2.copy(), b2.copy()

_, A1, _, y_hat = forward(X, W1, b1, W2, b2)
acc = np.mean((y_hat > 0.5) == y)        # 0.5 를 넘으면 "바깥 고리"로 예측한 것
print('크기 흐름: X', X.shape, '→ A1', A1.shape, '→ 예측', y_hat.shape, '(ch3 크기 조건 그대로)')
print(f'학습 전 정확도: {acc:.1%} — 동전 던지기 수준. 아직 아무것도 모릅니다.')

## 3. 손실 — 얼마나 틀렸는지 숫자 하나로

ch12 에서 배운 그대로: 학습하려면 먼저 "틀린 정도"를 숫자 하나로 만들어야 합니다.
**평균제곱오차(MSE)** — 예측과 정답의 차이를 제곱해서 평균낸 것:

$$L = \frac{1}{N}\sum_{i=1}^{N} (\hat{y}_i - y_i)^2 \qquad (N = 200,\ \textstyle\sum \text{는 "전부 더하라"})$$

- 제곱하는 이유: 부호를 없애고(+로 틀리든 −로 틀리든 틀린 건 틀린 것), 크게 틀릴수록 크게 벌점
- 어디서 봤죠? **ch10 최소제곱**의 "오차 제곱합"과 같은 아이디어 — 이름부터 재활용입니다

In [ ]:
def loss_fn(y_hat, y):
    return np.mean((y_hat - y) ** 2)     # 평균제곱오차 — ch10 최소제곱의 아이디어

print(f'학습 전 손실: {loss_fn(y_hat, y):.4f}  (이 숫자를 경사하강으로 굴려 내릴 겁니다)')

## 4. 역전파 — 책임을 거꾸로 배분하기

가중치 33개 각각에 "네가 손실에 얼마나 책임이 있나"(그래디언트, ch12)를 계산할 차례.
ch12 에서 봤듯 **연쇄법칙 = 출력에서 입력 쪽으로 기울기를 곱하며 전달**하는 것이고, 그 전달이 놀랍게도 전부 **행렬곱**입니다.

**밖에서 받아 적는 선물 두 줄** — 유도는 2부 미적분에서. 지금은 "그렇다더라"면 충분해요:

$$\sigma'(z) = \sigma(z)\,\bigl(1-\sigma(z)\bigr), \qquad \tanh'(z) = 1 - \tanh^2(z)$$

(프라임 $'$ = "기울기 함수" — ch12 의 $L'(w)$ 와 같은 표기)

이제 조립입니다. $\delta$(델타) = "그 층이 손실에 지는 책임"이라 부르면:

$$\delta_2 = \frac{2}{N}(\hat{y}-y) \odot \hat{y}(1-\hat{y}) \qquad\Rightarrow\qquad \nabla_{W_2} L = A_1^{\top}\,\delta_2$$

$$\delta_1 = (\delta_2\, W_2^{\top}) \odot (1 - A_1^2) \qquad\Rightarrow\qquad \nabla_{W_1} L = X^{\top}\,\delta_1$$

- $\odot$ = **같은 자리끼리** 곱하기(성분별 곱) — 행렬곱과 다른 것! 코드로는 그냥 `*`
- $\nabla_{W_2} L$ = "$W_2$ 로의 그래디언트" — ch12 의 $\nabla L$ 을 행렬 모양으로 쌓은 것
- **전치 $^\top$(ch3)의 재출연**: 순전파가 $A_1 W_2$ 였으니 거꾸로 갈 땐 $W_2^{\top}$.
  크기를 맞춰 보면 이 방향뿐입니다: $(200{\times}1)(1{\times}8) = 200{\times}8$ ✓

공식이 정말 맞는지는 ch12 방식 그대로 — **수치미분으로 검산**(gradient check)합니다.

In [ ]:
def backward(X, y, W2, A1, y_hat):
    """역전파 — 각 가중치의 그래디언트(책임량)를 출력 → 입력 순서로 계산."""
    N = len(X)
    delta2 = 2 * (y_hat - y) / N * y_hat * (1 - y_hat)   # 출력층의 책임 (선물 1: σ' = σ(1-σ))
    dW2 = A1.T @ delta2                                   # 전치(ch3)의 재출연!
    db2 = delta2.sum(axis=0, keepdims=True)
    delta1 = delta2 @ W2.T * (1 - A1 ** 2)                # 책임을 앞 층으로 전달 (선물 2: tanh')
    dW1 = X.T @ delta1
    db1 = delta1.sum(axis=0, keepdims=True)
    return dW1, db1, dW2, db2

# ── 검산: ch12 의 수치미분 그대로 (gradient check) ──
def loss_at(W1, b1, W2, b2):
    return loss_fn(forward(X, W1, b1, W2, b2)[3], y)

_, A1, _, y_hat = forward(X, W1, b1, W2, b2)
dW1, db1, dW2, db2 = backward(X, y, W2, A1, y_hat)

h = 1e-5
print('가중치      역전파         수치미분        차이')
for name, P, dP, idx in [('W1[0,0]', W1, dW1, (0, 0)), ('W2[3,0]', W2, dW2, (3, 0))]:
    P[idx] += h;      L_plus = loss_at(W1, b1, W2, b2)
    P[idx] -= 2 * h;  L_minus = loss_at(W1, b1, W2, b2)
    P[idx] += h                                    # 원상복구
    numeric = (L_plus - L_minus) / (2 * h)
    print(f'{name}  {dP[idx]:11.8f}  {numeric:11.8f}  {abs(dP[idx] - numeric):.1e}')

print('\n차이가 사실상 0 — 역전파 구현이 정확합니다. (검산 통과 ✅)')

## 5. 학습 — 경사하강 2,000걸음

ch12 의 갱신식을 모든 가중치에 동시에 적용할 뿐입니다:

$$W \;\leftarrow\; W - \eta\, \nabla_W L \qquad (\eta = \text{학습률})$$

우리 신경망의 가중치는 모두 **33개** ($W_1$ 16 + $b_1$ 8 + $W_2$ 8 + $b_2$ 1).
GPT 는 수천억 개지만 — 갱신식은 글자 하나 안 바뀝니다.

In [ ]:
eta = 0.8                                # 학습률 (ch12 — 너무 크면 발산, 작으면 느림)
losses, accs = [], []

for epoch in range(1, 2001):
    _, A1, _, y_hat = forward(X, W1, b1, W2, b2)
    dW1, db1, dW2, db2 = backward(X, y, W2, A1, y_hat)
    W1 -= eta * dW1;  b1 -= eta * db1    # 경사하강 — 이 네 줄이 '학습'의 전부
    W2 -= eta * dW2;  b2 -= eta * db2
    losses.append(loss_fn(y_hat, y))
    accs.append(np.mean((y_hat > 0.5) == y))
    if epoch % 400 == 0 or epoch == 1:
        print(f'{epoch:5}걸음  손실 {losses[-1]:.4f}  정확도 {accs[-1]:.1%}')

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(11, 3.8))
ax1.plot(losses, color='#6366f1')
ax1.set_xlabel('걸음 수 (epoch)'); ax1.set_ylabel('손실 L')
ax1.set_title('손실 — 골짜기로 내려가는 중 (ch12 그 곡선)')
ax2.plot(np.array(accs) * 100, color='#10b981')
ax2.set_xlabel('걸음 수 (epoch)'); ax2.set_ylabel('정확도 (%)')
ax2.set_title('정확도 — 동전 던지기에서 100% 로')
plt.tight_layout(); plt.show()

### 결정 경계 — 신경망이 그은 선 직접 보기

**결정 경계(decision boundary)** = 모델의 판단이 "안쪽"에서 "바깥"으로 넘어가는 경계선(확률 0.5인 곳).
평면 전체를 촘촘한 점으로 덮어 각 점의 판단을 색칠하면 경계가 드러납니다 — 그 점들의 묶음도 그냥 행렬이라, 한 번의 `forward` 로 끝나요.

In [ ]:
g1, g2 = np.meshgrid(np.linspace(-3.2, 3.2, 220), np.linspace(-3.2, 3.2, 220))
grid = np.c_[g1.ravel(), g2.ravel()]          # 48,400개 점 — 이것도 그냥 데이터 행렬 (48400×2)

fig, axes = plt.subplots(1, 2, figsize=(11, 5))
for ax, params, title in [
        (axes[0], (W1_0, b1_0, W2_0, b2_0), '학습 전 — 아무렇게나 그은 경계 (48%)'),
        (axes[1], (W1, b1, W2, b2),         '학습 후 — 스스로 찾아낸 원형 경계 (100%)')]:
    prob = forward(grid, *params)[3].reshape(g1.shape)
    ax.contourf(g1, g2, prob, levels=[0, 0.5, 1], colors=['#e0e7ff', '#fee2e2'])
    ax.contour(g1, g2, prob, levels=[0.5], colors='#334155', linewidths=1.5)
    ax.scatter(*inner.T, s=16, color='#4f46e5', label='안쪽 무리 (y=0)')
    ax.scatter(*outer.T, s=16, color='#f43f5e', label='바깥 고리 (y=1)')
    ax.set_aspect('equal'); ax.set_title(title); ax.legend(fontsize=8, loc='upper right')
plt.tight_layout(); plt.show()

print('아무도 "원을 그려라"라고 알려주지 않았습니다. 경사하강이 스스로 원형 경계를 찾아냈어요.')

## 6. 은닉층이 한 일 — 공간을 구부렸다

이상하지 않나요? 출력층이 하는 일은 $A_1 W_2 + b_2$ — 결국 **내적(ch2) 기준선 하나로 나누는 "직선적인" 판단**입니다. 그런데 어떻게 원형 경계가 나왔을까요?

답: **은닉층이 먼저 공간을 구부려서**(ch5 "행렬 = 공간 변형" + tanh 의 휘어짐),
두 무리를 **직선으로 나눌 수 있는 배치로 옮겨놨기** 때문입니다. 출력층은 옮겨진 공간에서 편하게 직선만 긋는 거죠.

은닉 표현 $A_1$ 은 8차원이라 눈으로 못 봅니다. 그래서 배운 도구로 눌러서 봐요:
- **가로축** = 출력층이 실제로 쓰는 축 — $A_1$ 과 $W_2$ 의 내적(ch2), 즉 $W_2$ 방향으로의 투영(ch10). 경계는 "점수 0"인 세로선 하나!
- **세로축** = 그 축과 수직(직교, ch10)인 성분 중 데이터가 가장 넓게 퍼진 방향 — PCA(ch8)

In [ ]:
def hidden_view(W1x, b1x, W2x, b2x):
    """은닉 표현(8차원)을 2차원으로 눌러서 보기 — 가로: 출력층 점수, 세로: 수직 성분의 PCA."""
    H = np.tanh(X @ W1x + b1x)                # 은닉 표현 (200×8)
    xs = (H @ W2x + b2x).ravel()              # 출력층의 점수 = W2 방향 내적 (0 이 경계)
    u = (W2x / np.linalg.norm(W2x)).ravel()   # W2 방향 단위벡터
    Hc = H - H.mean(axis=0)
    R = Hc - np.outer(Hc @ u, u)              # W2 방향 성분을 뺀 잔차 — u 와 수직 (ch10)
    vals, vecs = np.linalg.eigh(np.cov(R.T))  # 공분산행렬의 고유벡터 (ch8 PCA)
    ys = R @ vecs[:, -1]                      # 잔차 중 가장 넓게 퍼진 방향
    return xs, ys

fig, axes = plt.subplots(1, 2, figsize=(11, 4.2))
for ax, params, title in [
        (axes[0], (W1_0, b1_0, W2_0, b2_0), '학습 전 — 뒤섞여 있음'),
        (axes[1], (W1, b1, W2, b2),         '학습 후 — 세로선 하나로 갈라짐')]:
    xs, ys = hidden_view(*params)
    ax.scatter(xs[:n], ys[:n], s=16, color='#4f46e5', label='안쪽 무리')
    ax.scatter(xs[n:], ys[n:], s=16, color='#f43f5e', label='바깥 고리')
    ax.axvline(0, color='#334155', lw=1.5, ls='--', label='출력층의 경계 (점수 0)')
    ax.set_xlabel('출력층 점수 (W2 방향 내적)'); ax.set_ylabel('수직 성분의 PCA')
    ax.set_title(title); ax.legend(fontsize=8)
plt.tight_layout(); plt.show()

print('은닉층이 공간을 구부려 "직선으로 나눌 수 있는 문제"로 바꿔놨습니다.')
print('딥러닝이 층을 쌓는 이유가 이것 — 층마다 조금씩 더 나누기 좋은 공간으로 옮기는 것!')

---

### 정리 — 오늘 쓴 조각과 그 출처

| 캡스톤에서 한 일 | 쓴 개념 | 배운 곳 |
|---|---|---|
| 점 = 벡터, 점 200개 = 200×2 행렬 | 벡터 · 데이터 행렬 | ch1 · ch3 |
| 뉴런 하나의 계산 = 가중치와의 내적 | 내적 | ch2 |
| 층 하나 = $XW + b$, 크기 맞추기 | 행렬곱 | ch3 |
| 활성화 없으면 층이 접힌다 — $(XW_1)W_2 = X(W_1W_2)$ | 결합법칙 · 선형변환 | ch3 · ch5 |
| 은닉층 = 공간 구부리기 | 선형변환 + 비선형 | ch5 |
| 손실 = 오차 제곱의 평균 | 최소제곱 · 손실 | ch10 · ch12 |
| 역전파 = 전치행렬 곱의 연쇄 | 전치 · 연쇄법칙 | ch3 · ch12 |
| 검산 = 수치미분 | gradient check | ch12 |
| 학습 = $W \leftarrow W - \eta \nabla L$ | 경사하강 | ch12 |
| 은닉 표현 들여다보기 | 내적 · 투영 · PCA | ch2 · ch10 · ch8 |

**새로 받아 적은 것은 활성화 함수의 기울기 공식 두 줄뿐.**
그 두 줄이 "왜" 그런지 궁금해졌다면 — 그게 바로 2부 미적분의 출발점입니다.

### 🎓 캡스톤 수료 — 1부 완결

ch1 에서 "벡터가 뭐지?"로 시작해, 오늘 **신경망을 손수 만들어 학습**시켰습니다.
입력 → 순전파 → 손실 → 역전파 → 갱신. 이 다섯 단계는 GPT 같은 최신 AI 에서도 똑같이 돌아갑니다.
크기만 다를 뿐 — 수학은 오늘 여러분이 직접 쓴 그것입니다.

다음 여정 — **2부 미적분**: 오늘 "선물"로 받아 적은 기울기 공식을 직접 만들 수 있게 됩니다.